In [0]:
dbutils.widgets.text("catalog_name", "workspace")
dbutils.widgets.text("schema_name", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("table_name", "all")

catalog_name  = dbutils.widgets.get("catalog_name")
schema_name   = dbutils.widgets.get("schema_name")
silver_schema = dbutils.widgets.get("silver_schema")
table_name    = dbutils.widgets.get("table_name")

print("=" * 50)
print("PIPELINE PARAMETERS")
print("=" * 50)
print(f"Catalog       : {catalog_name}")
print(f"Bronze Schema : {schema_name}")
print(f"Silver Schema : {silver_schema}")
print(f"Table         : {table_name}")
print("=" * 50)

In [0]:

# Install Databricks DQX library
%pip install databricks-labs-dqx


In [0]:
dbutils.library.restartPython()

Re-read parameters after restart

In [0]:
# Must re-read widgets after restartPython()
catalog_name  = dbutils.widgets.get("catalog_name")
schema_name   = dbutils.widgets.get("schema_name")
silver_schema = dbutils.widgets.get("silver_schema")
table_name    = dbutils.widgets.get("table_name")

print(f"Parameters reloaded — Catalog: {catalog_name}, Bronze: {schema_name}, Silver: {silver_schema}")

In [0]:
active_tables = spark.sql(f"""
    SELECT table_name
    FROM {catalog_name}.{schema_name}.pipeline_control
    WHERE active_flag = 'Y'
    ORDER BY table_name
""")

print("=" * 50)
print("TABLES TO PROCESS THROUGH DQX")
print("=" * 50)
active_tables.show()
print(f"Total: {active_tables.count()} tables")

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog_name}.{schema_name}.dqx_execution_log (
        table_name      STRING,
        run_time        TIMESTAMP,
        total_records   LONG,
        passed_records  LONG,
        failed_records  LONG
    )
    USING DELTA
""")
print("✅ dqx_execution_log table ready")

dqx quality rules

In [0]:
rules_map = {
    "Invoice": [
        {"name": "InvoiceId_not_null",   "criticality": "error", "check": {"function": "is_not_null",      "arguments": {"column": "InvoiceId"}}},
        {"name": "CustomerId_not_null",  "criticality": "error", "check": {"function": "is_not_null",      "arguments": {"column": "CustomerId"}}},
        {"name": "InvoiceDate_not_null", "criticality": "error", "check": {"function": "is_not_null",      "arguments": {"column": "InvoiceDate"}}},
        {"name": "Total_not_null",       "criticality": "error", "check": {"function": "is_not_null",      "arguments": {"column": "Total"}}},
        {"name": "Total_range",          "criticality": "error", "check": {"function": "is_not_less_than", "arguments": {"column": "Total", "limit": 0}}},
        {"name": "InvoiceDate_valid",    "criticality": "warn",  "check": {"function": "is_valid_date",    "arguments": {"column": "InvoiceDate"}}},
    ],
    "InvoiceLine": [
        {"name": "InvoiceLineId_not_null","criticality": "error", "check": {"function": "is_not_null",      "arguments": {"column": "InvoiceLineId"}}},
        {"name": "InvoiceId_not_null",   "criticality": "error", "check": {"function": "is_not_null",      "arguments": {"column": "InvoiceId"}}},
        {"name": "TrackId_not_null",     "criticality": "error", "check": {"function": "is_not_null",      "arguments": {"column": "TrackId"}}},
        {"name": "UnitPrice_range",      "criticality": "error", "check": {"function": "is_not_less_than", "arguments": {"column": "UnitPrice",    "limit": 0}}},
        {"name": "Quantity_range",       "criticality": "error", "check": {"function": "is_not_less_than", "arguments": {"column": "Quantity",     "limit": 0}}},
    ],
    "Customer": [
        {"name": "CustomerId_not_null",  "criticality": "error", "check": {"function": "is_not_null",              "arguments": {"column": "CustomerId"}}},
        {"name": "FirstName_not_null",   "criticality": "error", "check": {"function": "is_not_null",              "arguments": {"column": "FirstName"}}},
        {"name": "LastName_not_null",    "criticality": "error", "check": {"function": "is_not_null",              "arguments": {"column": "LastName"}}},
        {"name": "Email_not_null",       "criticality": "error", "check": {"function": "is_not_null",              "arguments": {"column": "Email"}}},
        {"name": "Email_not_empty",      "criticality": "error", "check": {"function": "is_not_null_and_not_empty","arguments": {"column": "Email"}}},
    ],
    "Employee": [
        {"name": "EmployeeId_not_null",  "criticality": "error", "check": {"function": "is_not_null",   "arguments": {"column": "EmployeeId"}}},
        {"name": "FirstName_not_null",   "criticality": "error", "check": {"function": "is_not_null",   "arguments": {"column": "FirstName"}}},
        {"name": "LastName_not_null",    "criticality": "error", "check": {"function": "is_not_null",   "arguments": {"column": "LastName"}}},
        {"name": "BirthDate_valid",      "criticality": "warn",  "check": {"function": "is_valid_date", "arguments": {"column": "BirthDate"}}},
        {"name": "HireDate_valid",       "criticality": "warn",  "check": {"function": "is_valid_date", "arguments": {"column": "HireDate"}}},
    ],
    "Track": [
        {"name": "TrackId_not_null",     "criticality": "error", "check": {"function": "is_not_null",      "arguments": {"column": "TrackId"}}},
        {"name": "Name_not_null",        "criticality": "error", "check": {"function": "is_not_null",      "arguments": {"column": "Name"}}},
        {"name": "UnitPrice_range",      "criticality": "error", "check": {"function": "is_not_less_than", "arguments": {"column": "UnitPrice",    "limit": 0}}},
        {"name": "Milliseconds_range",   "criticality": "error", "check": {"function": "is_not_less_than", "arguments": {"column": "Milliseconds", "limit": 0}}},
    ],
    "Album": [
        {"name": "AlbumId_not_null",     "criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "AlbumId"}}},
        {"name": "Title_not_null",       "criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Title"}}},
        {"name": "ArtistId_not_null",    "criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "ArtistId"}}},
    ],
    "Artist": [
        {"name": "ArtistId_not_null",    "criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "ArtistId"}}},
        {"name": "Name_not_null",        "criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Name"}}},
    ],
}
print("✅ DQX rules defined for:", list(rules_map.keys()))

Define profiling helper functions

In [0]:
from pyspark.sql.functions import (
    col, sum as spark_sum, countDistinct,
    min as spark_min, max as spark_max,
    avg, to_date
)

def check_nulls(df, tbl_name):
    print(f"\n  [1] NULL VALUES — {tbl_name}")
    total_rows = df.count()
    null_exprs = [spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]
    null_counts = df.select(null_exprs).collect()[0].asDict()
    has_nulls = False
    for col_name, null_count in null_counts.items():
        pct  = round((null_count / total_rows) * 100, 1) if total_rows > 0 else 0
        flag = "⚠️ " if null_count > 0 else "✅"
        print(f"    {flag} {col_name}: {null_count} nulls ({pct}%)")
        if null_count > 0:
            has_nulls = True
    if not has_nulls:
        print("    ✅ No nulls found")
    return null_counts

def check_duplicates(df, tbl_name):
    print(f"\n  [2] DUPLICATE RECORDS — {tbl_name}")
    total_rows   = df.count()
    distinct_rows = df.dropDuplicates().count()
    dup_count    = total_rows - distinct_rows
    flag = "⚠️ " if dup_count > 0 else "✅"
    print(f"    {flag} {dup_count} duplicate rows out of {total_rows} total")
    return dup_count

def check_data_types(df, tbl_name):
    print(f"\n  [3] DATA TYPE CONSISTENCY — {tbl_name}")
    for c, dtype in df.dtypes:
        if "date" in c.lower():
            invalid = df.filter(
                col(c).isNotNull() & to_date(col(c)).isNull()
            ).count()
            flag = "⚠️ " if invalid > 0 else "✅"
            print(f"    {flag} {c} ({dtype}): {invalid} unparseable date values")
        else:
            print(f"    ✅ {c}: {dtype}")

def check_value_ranges(df, tbl_name):
    print(f"\n  [4] VALUE RANGES — {tbl_name}")
    numeric_cols = [c for c, t in df.dtypes
                    if t in ("int", "long", "double", "float", "decimal(10,2)")]
    if not numeric_cols:
        print("    ℹ️  No numeric columns")
        return
    range_exprs = []
    for c in numeric_cols:
        range_exprs += [
            spark_min(col(c)).alias(f"{c}_min"),
            spark_max(col(c)).alias(f"{c}_max"),
            avg(col(c)).alias(f"{c}_avg")
        ]
    ranges = df.select(range_exprs).collect()[0].asDict()
    for c in numeric_cols:
        mn   = ranges.get(f"{c}_min")
        mx   = ranges.get(f"{c}_max")
        av   = round(ranges.get(f"{c}_avg") or 0, 2)
        flag = "⚠️ " if (mn is not None and mn < 0) else "✅"
        print(f"    {flag} {c}: min={mn}, max={mx}, avg={av}")

def check_uniqueness(df, tbl_name):
    print(f"\n  [5] COLUMN UNIQUENESS — {tbl_name}")
    total_rows = df.count()
    for c in df.columns:
        distinct_count = df.select(countDistinct(col(c))).collect()[0][0]
        pct  = round((distinct_count / total_rows) * 100, 1) if total_rows > 0 else 0
        if "id" in c.lower() and distinct_count < total_rows:
            print(f"    ⚠️  {c}: {distinct_count} distinct / {total_rows} total ({pct}%) — possible duplicate IDs")
        else:
            print(f"    ✅ {c}: {distinct_count} distinct ({pct}%)")

print("✅ Profiling functions defined")

Profile + DQX validation

In [0]:
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.check_funcs import (
    is_not_null,
    is_not_less_than,
    is_valid_date,
    is_not_null_and_not_empty
)
from databricks.sdk import WorkspaceClient
from pyspark.sql.functions import col, to_date
from datetime import datetime

# Initialize DQEngine correctly
wc  = WorkspaceClient()
dqe = DQEngine(workspace_client=wc, spark=spark)
print("✅ DQEngine initialized")

validation_results = {}
success_tables = []
failed_tables  = []

print("=" * 50)
print("STARTING PROFILING + DQX VALIDATION")
print("=" * 50)

for row in active_tables.collect():
    tbl = row["table_name"]

    try:
        print(f"\n{'='*50}")
        print(f"  PROCESSING: {tbl}")
        print(f"{'='*50}")

        # ── Read from Bronze ──────────────────────────────────
        df_bronze     = spark.table(f"{catalog_name}.{schema_name}.{tbl}")
        total_records = df_bronze.count()
        print(f"\n  Bronze rows: {total_records}")

        # ── Data Profiling (all 5 checks) ─────────────────────
        check_nulls(df_bronze, tbl)
        check_duplicates(df_bronze, tbl)
        check_data_types(df_bronze, tbl)
        check_value_ranges(df_bronze, tbl)
        check_uniqueness(df_bronze, tbl)

        # ── DQX Validation ────────────────────────────────────
        print(f"\n  [DQX] Running validation for: {tbl}")
        if tbl in rules_map:
            good_df, bad_df = dqe.apply_checks_by_metadata_and_split(
            df_bronze, rules_map[tbl]
    )
        else:
            print(f"  ℹ️  No rules defined — passing all records through")
            good_df = df_bronze
            bad_df  = spark.createDataFrame([], df_bronze.schema)

        passed_records = good_df.count()
        failed_records = bad_df.count()
        print(f"  ✅ Passed : {passed_records}")
        print(f"  ❌ Failed : {failed_records}")

        # ── Log DQX execution metrics ─────────────────────────
        dqx_log = spark.createDataFrame([{
            "table_name"     : tbl,
            "run_time"       : datetime.now(),
            "total_records"  : total_records,
            "passed_records" : passed_records,
            "failed_records" : failed_records
        }])
        dqx_log.write.format("delta").mode("append") \
               .saveAsTable(f"{catalog_name}.{schema_name}.dqx_execution_log")
        print(f"  ✅ DQX metrics logged")

        # ── Store results for Silver load ─────────────────────
        validation_results[tbl] = {
            "good_df": good_df,
            "bad_df" : bad_df
        }
        success_tables.append(tbl)

    except Exception as e:
        failed_tables.append(tbl)
        print(f"\n  ❌ FAILED: {tbl} | {str(e)}")

print("\n" + "=" * 50)
print("PROFILING + VALIDATION COMPLETE")
print(f"✅ Ready for Silver load : {len(success_tables)} tables")
print(f"❌ Failed                : {len(failed_tables)} tables")
print("=" * 50)

delta tables into silver zone

In [0]:
from pyspark.sql.functions import trim, col

silver_success = []
silver_failed  = []

print("=" * 50)
print("LOADING TO SILVER")
print("=" * 50)

for tbl, results in validation_results.items():
    try:
        print(f"\n📤 Loading Silver: {tbl}")

        good_df   = results["good_df"]
        bad_df    = results["bad_df"]
        bad_count = bad_df.count()

        # ── Quarantine failed records ─────────────────────────
        if bad_count > 0:
            bad_df.write.format("delta").mode("append") \
                  .saveAsTable(f"{catalog_name}.{silver_schema}.quarantine_{tbl.lower()}")
            print(f"   ⚠️  {bad_count} records quarantined → quarantine_{tbl.lower()}")
        else:
            print(f"   ✅ No failed records — quarantine not needed")

        # ── Clean good records ────────────────────────────────
        clean_df = good_df

        # Trim all string columns
        for c, dtype in clean_df.dtypes:
            if dtype == "string":
                clean_df = clean_df.withColumn(c, trim(col(c)))

        # Fill nulls
        str_cols = [c for c, t in clean_df.dtypes if t == "string"]
        num_cols = [c for c, t in clean_df.dtypes
                    if t in ("int", "long", "double", "float", "decimal(10,2)")]
        if str_cols:
            clean_df = clean_df.fillna("Unknown", subset=str_cols)
        if num_cols:
            clean_df = clean_df.fillna(0, subset=num_cols)

        # ── Write to Silver ───────────────────────────────────
        silver_table = f"{catalog_name}.{silver_schema}.{tbl}"
        clean_df.write.format("delta").mode("overwrite") \
                .saveAsTable(silver_table)

        silver_count = spark.table(silver_table).count()
        print(f"   ✅ Silver rows : {silver_count} → {silver_table}")
        silver_success.append(tbl)

    except Exception as e:
        silver_failed.append(tbl)
        print(f"   ❌ FAILED: {tbl} | {str(e)}")

print("\n" + "=" * 50)
print("SILVER LOAD SUMMARY")
print("=" * 50)
print(f"✅ Loaded successfully : {len(silver_success)} tables")
print(f"❌ Failed              : {len(silver_failed)} tables")
for t in silver_success:
    print(f"  ✅ {t}")
if silver_failed:
    for t in silver_failed:
        print(f"  ❌ {t}")
print("=" * 50)

validate execution log

In [0]:
print("=" * 50)
print("DQX EXECUTION LOG")
print("=" * 50)
spark.sql(f"""
    SELECT table_name, total_records, passed_records, failed_records, run_time
    FROM {catalog_name}.{schema_name}.dqx_execution_log
    ORDER BY run_time DESC
""").show(truncate=False)

validating delta tables in silver zone

In [0]:
print("=" * 50)
print("SILVER TABLES IN UNITY CATALOG")
print("=" * 50)
spark.sql(f"SHOW TABLES IN {catalog_name}.{silver_schema}").show(truncate=False)

checking customer table

In [0]:
print("Spot check — Silver Customer (5 rows):")
spark.sql(f"SELECT * FROM {catalog_name}.{silver_schema}.Customer LIMIT 5").show()

checking invoice table

In [0]:
print("Spot check — Silver Invoice (5 rows):")
spark.sql(f"SELECT * FROM {catalog_name}.{silver_schema}.Invoice LIMIT 5").show()